# Dive.ai 데이터셋 활용 계획

**프로젝트**: Dive.ai — 시나리오, 캐릭터 빌더 및 인터랙티브 캐릭터 채팅 웹서비스  
**활용 데이터셋**:
- AI Hub 145: 다양한 문화콘텐츠 스토리 데이터 (3,561개 작품 / 7,122개 JSON 파일)
- AI Hub 70: 동아시아 고전 스토리 데이터 (816개 작품 / 25,434개 txt+json 파일)

---

이 노트북은 두 데이터셋의 구조를 Dive.ai의 핵심 기능(시나리오 빌더, 시나리오 트랜스포머, 로어북)과 개발 파이프라인(Phase 1~5)에 맞춰
구체적으로 어떻게 활용할지 정리한 계획서입니다.

---
## 0. 두 데이터셋 핵심 비교

| 항목 | 문화콘텐츠 스토리 (145) | 동아시아 고전 (70) |
|------|------------------------|--------------------|
| 원천 형식 | JSON (작품 단위) | TXT (단락 단위) |
| 카테고리 | 영화 / 시리즈 / 소설 / 만화 | 한국 / 중국 / 일본 |
| 분석 단위 | 작품 전체 + 씬(unit) 분해 | 단락 단위 |
| 스토리 구조 | 스토리헬퍼 15단계 stage 레이블 | 없음 (내용 분류 중심) |
| 캐릭터 처리 | 익명화 (C001~Cnnn) | 실명 그대로 |
| 감정·행동 레이블 | 문장 단위 (act / emotion) | 단락 단위 (감정 / 행동) |
| 장르 레이블 | 작품 단위 복수 장르 | 단락 단위 장르 |
| 서사 모티프 | 작품 motif + 씬 unit_motif | 단락 사건/모티프 |
| Dive.ai 주 역할 | **모든 장르의 서사 구조 학습** (시나리오 빌더·트랜스포머·로어북·캐릭터 채팅) | **고전 장르 선택 시에만** 세계관·어조 보강 |

> **핵심 전략**: 두 데이터셋의 역할을 명확히 분리합니다.  
> 문화콘텐츠 데이터가 **모든 장르의 주(主) 데이터**입니다. 시나리오 구조, 캐릭터 페르소나, 로어북, 채팅 모두 문화콘텐츠 데이터를 기본으로 동작합니다.  
> 동아시아 고전 데이터는 유저가 **"조선", "중국 무협", "일본 설화" 등 고전 장르를 명시적으로 선택했을 때만** 보조적으로 참조합니다.

```python
# 고전 장르 여부를 판별하는 플래그 — 전 파이프라인에서 공통 사용
CLASSIC_GENRES = {"조선", "중국무협", "일본설화", "한국설화", "동아시아고전"}

def is_classic_genre(background: str) -> bool:
    """유저가 선택한 배경이 고전 장르인지 판별."""
    return any(tag in background for tag in CLASSIC_GENRES)
```


---
## 1. Phase 1 — 데이터 전처리 및 지식 베이스 구축

### 1-1. 문화콘텐츠 스토리 데이터 전처리

#### (1) 장르·테마·모티프 메타데이터 추출
RAG의 검색 품질은 문서에 붙어 있는 메타데이터에 크게 좌우됩니다.  
라벨링 JSON에서 아래 필드를 추출하여 각 청크의 메타데이터로 보존합니다.

```python
# 라벨링 JSON → 메타데이터 딕셔너리 변환 예시
def extract_meta(label: dict) -> dict:
    return {
        "id":             label["id"],
        "type":           label["type"],          # movie | series | novel | comic
        "genre":          label["genre"],          # ["스릴러", "드라마"] 등
        "theme":          label["theme"],          # "발견"
        "motif":          label["motif"],          # "갑작스런 사고"
        "main_character": label["main_character"], # 주인공 성격
        "structure":      label["structure"],      # "스토리헬퍼 15단계"
    }
```

#### (2) 씬(unit) 단위 청킹 전략
문화콘텐츠 데이터는 이미 **씬(unit) 단위**로 분절되어 있어 청킹에 최적화된 구조입니다.  
씬 하나를 청크 하나로 처리하고, `stage` + `unit_motif` + `storyline`을 메타데이터로 붙입니다.

```python
from langchain.schema import Document

def units_to_documents(label: dict) -> list[Document]:
    """라벨링 JSON의 units 배열을 LangChain Document 리스트로 변환."""
    docs = []
    work_meta = extract_meta(label)
    for unit in label["units"]:
        # 씬의 모든 대사/서술을 하나의 텍스트로 합치기
        text_parts = [s["content"] for s in unit["story_scripts"]]
        full_text = " ".join(text_parts)
        
        meta = {
            **work_meta,
            "unit_id":    unit["id"],
            "stage":      unit["stage"],       # "Opening Salvo" 등 스토리헬퍼 단계
            "unit_motif": unit["unit_motif"],  # 씬 모티프
            "storyline":  unit["storyline"],   # 씬 요약
        }
        docs.append(Document(page_content=full_text, metadata=meta))
    return docs
```

#### (3) 씬 내부 감정·행동 레이블 활용
`story_scripts[]` 안의 `act` / `emotion` 필드는 캐릭터 페르소나 프롬프트 설계에 직접 활용합니다.  
같은 씬에서 특정 캐릭터가 어떤 감정 상태로 어떤 행동을 했는지 패턴을 학습할 수 있습니다.

```python
def extract_character_acts(unit: dict) -> list[dict]:
    """씬 내 캐릭터별 감정-행동 패턴 추출."""
    return [
        {
            "character": s["character"],
            "act":       s["act"],
            "emotion":   s["emotion"],
            "type":      s["type"],      # narrative | dialogue
            "location":  s["location"],
        }
        for s in unit["story_scripts"]
    ]
```

### 1-2. 동아시아 고전 데이터 전처리

#### (1) 단락 단위 청킹
동아시아 고전 데이터는 파일 하나가 곧 단락 하나이므로 **파일 = 청크** 구조입니다.  
txt 원문과 `_info.json` 레이블을 결합하여 Document를 생성합니다.

```python
def classic_to_document(src_text: str, label: dict, file_stem: str) -> Document:
    """동아시아 고전 단락을 LangChain Document로 변환."""
    country_map = {"01": "한국", "02": "중국", "03": "일본"}
    classification = label["내용분류"]
    
    meta = {
        "file_stem":   file_stem,
        "country":     country_map.get(label["country"], label["country"]),
        "genre":       classification["장르"],
        "motif":       classification["사건/모티프"],
        "emotion":     classification["감정"],
        "action":      classification["행동"],
        "space":       classification["공간"],
        "character":   classification["캐릭터"],
        "occupation":  classification["직업/신분"],
        "keywords":    label["주제어"],
        "summary":     label["주제문"],
        "source":      "동아시아고전",
    }
    return Document(page_content=src_text, metadata=meta)
```

#### (2) 국가별 세계관 인덱스 분리 구성
유저가 장르 입력 시 "조선시대", "중국 무협", "일본 설화" 등 지역 배경을 선택할 수 있도록  
ChromaDB 컬렉션을 국가별로 분리하거나 메타데이터 필터링을 사용합니다.

```python
# 컬렉션 분리 방식 (검색 속도 우선)
collections = {
    "korea_classic":  chroma_client.get_or_create_collection("korea_classic"),
    "china_classic":  chroma_client.get_or_create_collection("china_classic"),
    "japan_classic":  chroma_client.get_or_create_collection("japan_classic"),
}

# 또는 단일 컬렉션 + 메타데이터 필터 방식 (유연성 우선)
results = vector_db.similarity_search(
    query="퇴마사 이야기",
    filter={"country": "한국", "genre": {"$in": ["설화", "무협"]}},
    k=5
)
```

### 1-3. 통합 Vector DB 설계

```
[Vector DB 컬렉션 구조]

① cultural_content_scenes    — 문화콘텐츠 씬 단위 청크 (stage/motif/genre 메타)
② cultural_content_works     — 문화콘텐츠 작품 단위 요약 (concept/conflict/theme 메타)
③ classic_korea              — 한국 고전 단락
④ classic_china              — 중국 고전 단락
⑤ classic_japan              — 일본 고전 단락
⑥ lorebook_events            — 로어북 사건 로그 (대화 중 실시간 업데이트)
```

> **권장 스택**: ChromaDB (로컬·프로토타입) → Pinecone (서비스 배포 시)  
> **임베딩 모델**: `text-embedding-3-small` (OpenAI) 또는 `KoSimCSE` (한국어 특화)

---
## 2. Phase 2 — 시나리오 빌더 & 로어북 엔진

### 2-1. 시나리오 빌더: RAG 기반 서사 패턴 소환

유저가 **"조선시대 퇴마사 복수극"** 같은 키워드를 입력하면,  
**문화콘텐츠 데이터를 기본**으로 서사 구조를 검색하고,  
고전 장르(`is_classic_genre`)일 때만 동아시아 고전 데이터를 추가로 참조합니다.

```python
def build_scenario_prompt(user_input: str, genre: str, background: str) -> str:
    """유저 입력 + RAG 검색 결과로 시나리오 생성 프롬프트 구성."""

    # ① 문화콘텐츠에서 동일 장르의 흥행 서사 구조 검색 (항상 실행)
    structure_refs = vector_db["cultural_content_scenes"].similarity_search(
        query=user_input,
        filter={"genre": {"$in": [genre]}},
        k=5
    )
    stage_examples = [
        f"[{doc.metadata['stage']}] {doc.metadata['storyline']}"
        for doc in structure_refs
    ]

    # ② 고전 장르일 때만 동아시아 고전 데이터 추가 참조
    classic_section = ""
    if is_classic_genre(background):
        country = background_to_country(background)  # "조선" → "한국" 등
        classic_refs = vector_db[f"classic_{country}"].similarity_search(
            query=user_input, k=3
        )
        classic_section = (
            "\n[세계관·분위기 참고 (동아시아 고전)]\n"
            + "\n".join([doc.metadata["summary"] for doc in classic_refs])
        )

    prompt = f"""
당신은 전문 시나리오 작가입니다. 아래 참고 자료를 바탕으로 사용자의 소재를 기승전결이 뚜렷한
인터랙티브 시나리오로 확장하세요.

[사용자 소재]
{user_input}

[장르별 흥행 서사 구조 참고 (스토리헬퍼 15단계)]
{chr(10).join(stage_examples)}
{classic_section}

[출력 형식]
- 제목, 주제(theme), 핵심 갈등(conflict), 주인공 성격
- 기승전결 줄거리 (각 단계 2~3문장)
- 주요 등장인물 3인의 페르소나 (이름, 성격, 목표, 말투 특징)
"""
    return prompt
```

### 2-2. 스토리헬퍼 15단계 → 분기점 매핑

문화콘텐츠 데이터의 `units[].stage` 값은 스토리헬퍼 15단계 영문 이름으로 레이블되어 있습니다.  
Scenario Transformer는 이 단계를 분기점 후보로 자동 인식합니다.

```python
# 스토리헬퍼 15단계 중 갈등 분기에 적합한 stage 정의
BRANCH_STAGES = {
    "Inciting Incident":    "발단 — 사건의 시작, 첫 선택",
    "Point of No Return":   "돌아올 수 없는 지점 — 결정적 선택",
    "Midpoint":             "중간점 — 반전 또는 가치관 충돌",
    "Dark Night of Soul":   "절망의 밤 — 포기 or 극복 선택",
    "Climax":               "클라이맥스 — 최종 대결 분기",
}

def find_branch_candidates(label: dict) -> list[dict]:
    """작품에서 분기점 후보 씬 추출."""
    return [
        {
            "unit_id":   unit["id"],
            "stage":     unit["stage"],
            "storyline": unit["storyline"],
            "motif":     unit["unit_motif"],
            "branch_desc": BRANCH_STAGES.get(unit["stage"], "")
        }
        for unit in label["units"]
        if unit["stage"] in BRANCH_STAGES
    ]
```

### 2-3. 로어북 자동 구축

로어북 초안은 **문화콘텐츠 데이터를 기본**으로 생성합니다.  
고전 장르일 때만 동아시아 고전 데이터의 `공간` / `캐릭터` / `주제어`를 추가로 보강합니다.

```python
def generate_lorebook_draft(label: dict, background: str) -> dict:
    """로어북 초안 생성. 항상 문화콘텐츠 데이터 기반, 고전 장르일 때만 고전 데이터 보강."""

    # ① 기본: 문화콘텐츠 라벨링 데이터에서 추출 (항상 실행)
    draft = {
        "scenario":  label["title"],
        "lorebook_draft": {
            "핵심 갈등":   label.get("conflict", ""),
            "서사 모티프": label.get("motif", ""),
            "등장인물":    label.get("characters", []),
            "배경 공간":   list({
                s["location"]
                for unit in label["units"]
                for s in unit["story_scripts"]
                if s.get("location")
            }),
        }
    }

    # ② 고전 장르일 때만 고전 데이터로 세계관 보강
    if is_classic_genre(background):
        country = background_to_country(background)
        classic_docs = vector_db[f"classic_{country}"].similarity_search(
            query=label["concept"], k=3
        )
        classic_spaces   = {m for doc in classic_docs for m in doc.metadata.get("space", [])}
        classic_chars    = {m for doc in classic_docs for m in doc.metadata.get("character", [])}
        classic_keywords = {m for doc in classic_docs for m in doc.metadata.get("keywords", [])}

        draft["lorebook_draft"]["배경 공간"]   += list(classic_spaces)
        draft["lorebook_draft"]["고전 인물 유형"] = list(classic_chars)
        draft["lorebook_draft"]["고전 키워드"]  = list(classic_keywords)

    return draft
```


---
## 3. Phase 3 — 캐릭터 페르소나 및 채팅 최적화

### 3-1. 문화콘텐츠 데이터 → 캐릭터 페르소나 시스템 프롬프트 설계

`story_scripts[]`의 `act` + `emotion` 레이블을 집계하면 캐릭터별 행동-감정 프로파일을 만들 수 있습니다.  
이를 캐릭터 시스템 프롬프트의 **"성격과 반응 패턴"** 섹션에 반영합니다.

```python
from collections import Counter

def build_character_persona(label: dict, char_id: str) -> dict:
    """특정 캐릭터의 감정·행동 분포로 페르소나 프로파일 생성."""
    acts, emotions = [], []

    for unit in label["units"]:
        for s in unit["story_scripts"]:
            if char_id in s.get("character", []):
                if s.get("act"):     acts.append(s["act"])
                if s.get("emotion"): emotions.append(s["emotion"])

    top_acts     = [a for a, _ in Counter(acts).most_common(5)]
    top_emotions = [e for e, _ in Counter(emotions).most_common(5)]

    return {
        "character_id":      char_id,
        "personality":       label.get("main_character", ""),
        "dominant_acts":     top_acts,
        "dominant_emotions": top_emotions,
        "conflict_role":     label.get("conflict", ""),
    }


def persona_to_system_prompt(persona: dict, scenario_context: str) -> str:
    """페르소나 딕셔너리를 채팅용 시스템 프롬프트로 변환."""
    return f"""
당신은 아래 설정을 가진 캐릭터입니다. 항상 이 성격과 말투를 유지하세요.

[캐릭터 설정]
- 성격 유형: {persona['personality']}
- 주요 행동 패턴: {', '.join(persona['dominant_acts'])}
- 주요 감정 상태: {', '.join(persona['dominant_emotions'])}
- 서사에서의 역할: {persona['conflict_role']}

[현재 시나리오 맥락]
{scenario_context}

[대화 규칙]
1. 캐릭터의 감정 상태를 대사와 행동 묘사로 자연스럽게 표현하세요.
2. 시나리오 맥락을 벗어난 발언을 하지 마세요.
3. 유저의 선택에 따라 감정과 태도가 변화할 수 있습니다.
"""
```

### 3-2. 캐릭터 말투·어조 스타일

캐릭터 말투는 **문화콘텐츠 데이터의 `story_scripts[].type` / `act` / `emotion`** 패턴을 기반으로 합니다.  
동아시아 고전의 대화 예시는 유저가 **고전 장르를 명시적으로 선택했을 때만** few-shot으로 추가합니다.

```python
def get_dialogue_fewshot(background: str, dialogue_type: str, k: int = 3) -> str:
    """고전 장르일 때만 고전 대화 예시를 few-shot으로 반환. 비고전은 빈 문자열."""
    if not is_classic_genre(background):
        return ""  # 비고전 장르 → 고전 데이터 참조 안 함

    country = background_to_country(background)
    results = vector_db[f"classic_{country}"].similarity_search(
        query=dialogue_type,
        filter={"대화형태": {"$in": [dialogue_type]}},
        k=k
    )
    examples = [doc.page_content[:200] for doc in results]
    return "\n[고전 말투 참고 예시]\n" + "\n".join(examples) if examples else ""

# 사용 예시
# 현대 배경 → fewshot = "" (고전 데이터 미참조)
# 조선 배경 → fewshot = "\n[고전 말투 참고 예시]\n..."
fewshot = get_dialogue_fewshot(background, "명령")
```


---
## 4. Phase 4 — 시나리오 트랜스포머 & 멀티모달 연동

### 4-1. 선형 시나리오 → 인터랙티브 분기점 변환

문화콘텐츠 데이터의 흥행작에서 동일 `stage`(예: `Climax`) 씬들을 모아  
"이 지점에서 어떤 선택지가 등장하는가?" 패턴을 학습합니다.

```python
def build_branch_prompt(current_stage: str, storyline: str, genre: list) -> str:
    """현재 stage + 줄거리 맥락으로 분기 선택지 생성 프롬프트 구성."""

    # 동일 stage, 동일 장르의 흥행 씬 예시 검색
    refs = vector_db["cultural_content_scenes"].similarity_search(
        query=storyline,
        filter={
            "stage": current_stage,
            "genre": {"$in": genre}
        },
        k=4
    )

    ref_text = "\n".join([
        f"- [{doc.metadata['unit_motif']}] {doc.metadata['storyline']}"
        for doc in refs
    ])

    return f"""
현재 서사 단계: {current_stage}
현재 상황: {storyline}

[참고: 동일 단계의 흥행 씬 예시]
{ref_text}

위 상황에서 유저가 선택할 수 있는 분기 선택지 3개를 생성하세요.
각 선택지는 서사 모티프가 달라야 하며, 선택에 따라 이후 스토리가 뚜렷이 달라지도록 하세요.
형식: [선택지 A / B / C] + 각 선택이 초래하는 결과 힌트(1줄)
"""
```

### 4-2. 이미지 생성 프롬프트

이미지 프롬프트는 **문화콘텐츠 데이터의 `location` / `unit_motif`** 를 기본으로 구성합니다.  
고전 장르일 때만 동아시아 고전 데이터의 `공간` / `캐릭터` 키워드를 추가로 반영합니다.

```python
# 고전 장르 전용 시각 키워드 맵 (고전 장르 선택 시에만 사용)
CLASSIC_SPACE_VISUAL_MAP = {
    "궁":  "royal palace, hanok architecture, ornate wooden pillars",
    "전장": "battlefield, dust and smoke, warriors in armor",
    "산":  "misty mountain, ancient forest, stone path",
    "바다": "stormy sea, wooden sailing ship, moonlit waves",
    "사원": "Buddhist temple, incense smoke, lantern light",
}

CLASSIC_CHARACTER_VISUAL_MAP = {
    "왕족/귀족": "noble costume, silk hanbok, crown or gat hat",
    "승려":      "Buddhist monk robe, shaved head, prayer beads",
    "요괴":      "supernatural creature, glowing eyes, ethereal aura",
    "무사/장수": "armor, sword, fierce expression",
}

def build_image_prompt(
    location: str, unit_motif: str, emotion: str,
    background: str,
    classic_space: str = "", classic_characters: list = []
) -> str:
    """이미지 생성 프롬프트 구성.
    - 기본: 문화콘텐츠 데이터의 location + unit_motif 사용
    - 고전 장르일 때만 고전 공간·캐릭터 키워드 추가
    """
    # 기본 프롬프트 (항상 적용)
    base = f"{location}, mood: {emotion}, theme: {unit_motif}, cinematic lighting, detailed illustration"

    # 고전 장르일 때만 고전 키워드 보강
    if is_classic_genre(background) and classic_space:
        visual_space = CLASSIC_SPACE_VISUAL_MAP.get(classic_space, classic_space)
        visual_chars = ", ".join([CLASSIC_CHARACTER_VISUAL_MAP.get(c, c) for c in classic_characters])
        return f"{base}, {visual_space}, {visual_chars}, East Asian art style"

    return base
```


---
## 5. Phase 5 — 서사 일관성 테스트

### 5-1. 문화콘텐츠 데이터로 일관성 검증 케이스 구성

실제 흥행 작품의 씬 연결 구조(`units[].next_id`)를 활용하여  
"이전 씬 → 현재 씬" 전환이 자연스러운지 검증하는 골든 셋(golden set)을 만듭니다.

```python
def build_consistency_test_set(label: dict, n: int = 10) -> list[dict]:
    """연속된 씬 쌍으로 일관성 테스트 케이스 구성."""
    unit_map = {u["id"]: u for u in label["units"]}
    test_cases = []

    for unit in label["units"][:n]:
        next_id = unit.get("next_id")
        if next_id and next_id in unit_map:
            test_cases.append({
                "prev_stage":     unit["stage"],
                "prev_storyline": unit["storyline"],
                "next_stage":     unit_map[next_id]["stage"],
                "next_storyline": unit_map[next_id]["storyline"],
                "genre":          label["genre"],
                "expected_flow":  "natural",  # 기준값
            })
    return test_cases

# 활용: AI가 생성한 씬 전환과 실제 흥행작의 씬 전환을 비교하여
#       hallucination / 설정 충돌 여부를 자동 검증
```

### 5-2. 동아시아 고전 데이터로 세계관 충돌 감지

`주제문` 필드를 시맨틱 검색하여 대화 중 생성된 문장이 설정된 세계관과 충돌하는지 검사합니다.

```python
def check_world_consistency(generated_text: str, lorebook_collection, threshold: float = 0.3) -> bool:
    """생성된 문장이 로어북 세계관과 충돌하는지 코사인 유사도로 검사."""
    results = lorebook_collection.similarity_search_with_score(
        query=generated_text, k=3
    )
    # 가장 유사한 로어북 항목과의 거리가 임계값 이상이면 충돌 의심
    min_score = min(score for _, score in results)
    return min_score < threshold  # True = 일관성 있음
```

---
## 6. 데이터셋 활용 요약 매핑

> 동아시아 고전 데이터는 유저가 **고전 장르(조선/중국무협/일본설화 등)를 명시적으로 선택했을 때만** 참조됩니다.

| Dive.ai 기능 | 문화콘텐츠 (145) 활용 필드 | 동아시아 고전 (70) 활용 필드 |
|---|---|---|
| **시나리오 빌더** | `genre`, `theme`, `motif`, `concept`, `units[].stage`, `units[].storyline` | ⚡ 고전 장르 시: `장르`, `사건/모티프`, `주제문` |
| **캐릭터 생성** | `main_character`, `conflict`, `story_scripts[].act`, `story_scripts[].emotion` | ⚡ 고전 장르 시: `캐릭터`, `직업/신분`, `인물관계` |
| **시나리오 트랜스포머** | `units[].stage` (BRANCH_STAGES 매핑), `units[].unit_motif`, `next_id` 연결 구조 | — (미사용) |
| **로어북 자동 생성** | `characters[]`, `conflict`, `motif`, `story_scripts[].location` | ⚡ 고전 장르 시: `공간`, `캐릭터`, `직업/신분`, `주제어` 보강 |
| **캐릭터 채팅** | `story_scripts[].emotion`, `story_scripts[].act` | ⚡ 고전 장르 시: `대화형태`, `감정` (few-shot 말투 예시) |
| **이미지 생성** | `units[].unit_motif`, `story_scripts[].location` | ⚡ 고전 장르 시: `공간`, `캐릭터` 키워드 보강 |
| **일관성 테스트** | `units[].next_id` 연결 구조 (골든셋) | — (미사용) |

> ⚡ 표시는 **고전 장르 선택 시에만** 동작하는 보조 참조입니다.

---

## 7. 데이터 활용 시 주의사항

1. **캐릭터 익명화 처리 (문화콘텐츠 데이터)**  
   `C001`, `C002` 같은 익명 ID는 RAG 검색 청크에 그대로 포함될 경우 이상한 텍스트가 생성됩니다.  
   임베딩 전에 `C\d+`를 역할 기반 명칭("주인공", "조력자", "악당")으로 치환하거나  
   또는 `storyline` / `concept` 같은 요약 필드만 임베딩에 사용하는 전략을 권장합니다.

2. **동아시아 고전의 단락 길이 편차**  
   단락 txt 파일의 길이가 수십 자에서 수천 자까지 다양합니다.  
   토큰 기반 분할(예: 512 토큰 단위)을 추가 적용하거나, 너무 짧은 단락(50자 미만)은 필터링을 권장합니다.

3. **라이선스 및 이용 조건**  
   AI Hub 데이터셋은 비상업적 연구·개발 목적의 이용 조건이 있습니다.  
   서비스 상업화 시 AI Hub의 데이터 이용 정책을 반드시 재확인하세요.

4. **임베딩 언어 최적화**  
   두 데이터셋 모두 한국어 텍스트 중심이므로 범용 영어 임베딩 모델보다  
   한국어 특화 모델(`KoSimCSE`, `KLUE-RoBERTa` 기반 임베딩)의 검색 품질이 높을 수 있습니다.  
   OpenAI `text-embedding-3-small`은 다국어 성능이 양호하여 초기 프로토타입에 적합합니다.
